# コーン終端と z 軸方向の強度比較

`260619_1348` と `260621_1559` の field map から、コーン終端 `z = base_z + l_cone` の径方向平均強度と、`x=0, y=0` の z 軸方向強度を比較します。

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
VENV_DIR = (ROOT.parent / ".venv").resolve()
VENV_PYTHON = VENV_DIR / "bin" / "python"
VENV_SITE_PACKAGES = VENV_DIR / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"

if not VENV_DIR.exists():
    raise RuntimeError(f".venv が見つかりません: {VENV_DIR}")
if not VENV_SITE_PACKAGES.exists():
    raise RuntimeError(f"site-packages が見つかりません: {VENV_SITE_PACKAGES}")

# Jupyter 側で .venv の kernel を選べない場合でも、同じ Python 3.12 なら .venv のパッケージを優先して読む。
if str(VENV_SITE_PACKAGES) not in sys.path:
    sys.path.insert(0, str(VENV_SITE_PACKAGES))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-comparison-cache")

print(f"current python : {sys.executable}")
print(f"target .venv   : {VENV_PYTHON}")
print(f"site-packages  : {VENV_SITE_PACKAGES}")

: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("numpy     ", np.__version__)
print("matplotlib", plt.matplotlib.__version__)

In [ ]:
RESULT_DIRS = [Path("260619_1348"), Path("260621_1559")]
LABELS = [path.name for path in RESULT_DIRS]
OUTDIR = Path("comparison_profiles")
OUTDIR.mkdir(exist_ok=True)

RESULT_DIRS, OUTDIR

In [ ]:
def parse_geometry(path: Path) -> dict[str, float]:
    geometry = {}
    for line in path.read_text().splitlines():
        if "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.split("[", 1)[0].strip()
        token = value.split()[0]
        try:
            geometry[key] = float(token)
        except ValueError:
            pass
    return geometry


def parse_xy_index(path: Path):
    rows = []
    for line in path.read_text().splitlines():
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        index, z_value, region = line.split()[:3]
        rows.append((int(index), float(z_value), region))
    return rows


def choose_cone_end_xy_file(result_dir: Path):
    geometry = parse_geometry(result_dir / "geometry_summary.txt")
    cone_end_z = geometry["base_z"] + geometry["l_cone"]
    rows = parse_xy_index(result_dir / "xy_plane_index_to_z.txt")
    index, z_value, region = min(rows, key=lambda row: abs(row[1] - cone_end_z))
    return result_dir / f"field_map_xy_z{index:03d}.bin", z_value, cone_end_z, geometry


def read_field_map(path: Path):
    raw = np.fromfile(path, dtype="<f8")
    if raw.size % 4 != 0:
        raise ValueError(f"{path} は x, y, z, intensity の4平面に分けられません")
    n_points = raw.size // 4
    x, y, z, intensity = np.split(raw, 4)

    reset = np.flatnonzero(np.diff(x) <= 0)
    if reset.size == 0:
        raise ValueError(f"{path}: x 座標からグリッド幅を推定できません")
    nx = int(reset[0] + 1)
    ny = int(n_points // nx)
    shape = (ny, nx)
    return {
        "path": path,
        "nx": nx,
        "ny": ny,
        "x": x.reshape(shape),
        "y": y.reshape(shape),
        "z": z.reshape(shape),
        "intensity": intensity.reshape(shape),
    }


def radial_profile(field_map, max_radius=None, bin_width=None):
    x = field_map["x"]
    y = field_map["y"]
    intensity = field_map["intensity"]
    finite = np.isfinite(intensity)
    radius = np.hypot(x, y)

    if max_radius is None:
        max_radius = np.nanmax(radius[finite])
    if bin_width is None:
        x_axis = x[0, :]
        bin_width = np.min(np.diff(x_axis)[np.diff(x_axis) > 0])

    bins = np.arange(0.0, max_radius + bin_width, bin_width)
    bin_index = np.digitize(radius[finite], bins) - 1
    valid = (bin_index >= 0) & (bin_index < len(bins) - 1)
    bin_index = bin_index[valid]
    values = intensity[finite][valid]

    counts = np.bincount(bin_index, minlength=len(bins) - 1)
    sums = np.bincount(bin_index, weights=values, minlength=len(bins) - 1)
    sums_sq = np.bincount(bin_index, weights=values * values, minlength=len(bins) - 1)

    means = np.full(len(bins) - 1, np.nan)
    stds = np.full(len(bins) - 1, np.nan)
    nonzero = counts > 0
    means[nonzero] = sums[nonzero] / counts[nonzero]
    variance = sums_sq[nonzero] / counts[nonzero] - means[nonzero] ** 2
    stds[nonzero] = np.sqrt(np.maximum(variance, 0.0))
    centers = 0.5 * (bins[:-1] + bins[1:])

    if np.any(nonzero):
        last = np.flatnonzero(nonzero)[-1] + 1
        centers, means, stds, counts = centers[:last], means[:last], stds[:last], counts[:last]
    return centers, means, stds, counts


def axis_profile(field_map):
    x_axis = field_map["x"][0, :]
    z_axis = field_map["z"][:, 0]
    intensity = field_map["intensity"]
    return z_axis, np.array([np.interp(0.0, x_axis, row) for row in intensity])

In [ ]:
profiles = []

for result_dir, label in zip(RESULT_DIRS, LABELS):
    xy_file, xy_z, cone_end_z, geometry = choose_cone_end_xy_file(result_dir)
    xy_map = read_field_map(xy_file)
    xz_map = read_field_map(result_dir / "field_map_xz.bin")

    r, radial_mean, radial_std, radial_count = radial_profile(
        xy_map,
        max_radius=geometry["r_pipe"],
    )
    z, axis_i = axis_profile(xz_map)

    profiles.append({
        "label": label,
        "result_dir": result_dir,
        "xy_file": xy_file,
        "xy_z": xy_z,
        "cone_end_z": cone_end_z,
        "geometry": geometry,
        "r": r,
        "radial_mean": radial_mean,
        "radial_std": radial_std,
        "radial_count": radial_count,
        "z": z,
        "axis_i": axis_i,
    })

for profile in profiles:
    print(f"{profile['label']}: cone end z={profile['cone_end_z']:.6f} m, selected {profile['xy_file']} at z={profile['xy_z']:.6f} m")

In [ ]:
radial_columns = [profiles[0]["r"], profiles[0]["r"] * 1000.0]
radial_header = ["r_m", "r_mm"]
for profile in profiles:
    radial_columns += [profile["radial_mean"], profile["radial_std"], profile["radial_count"]]
    radial_header += [f"{profile['label']}_mean", f"{profile['label']}_std", f"{profile['label']}_count"]

radial_table = np.column_stack(radial_columns)
np.savetxt(
    OUTDIR / "radial_profile_cone_end.csv",
    radial_table,
    delimiter=",",
    header=",".join(radial_header),
    comments="",
)

axis_columns = [profiles[0]["z"], profiles[0]["z"] * 1000.0, *[profile["axis_i"] for profile in profiles]]
axis_header = ["z_m", "z_mm", *[f"{profile['label']}_intensity" for profile in profiles]]
axis_table = np.column_stack(axis_columns)
np.savetxt(
    OUTDIR / "axis_profile_z.csv",
    axis_table,
    delimiter=",",
    header=",".join(axis_header),
    comments="",
)

print(OUTDIR / "radial_profile_cone_end.csv")
print(OUTDIR / "axis_profile_z.csv")

In [ ]:
plt.figure(figsize=(8, 5))
for profile in profiles:
    plt.plot(profile["r"] * 1000.0, profile["radial_mean"], label=profile["label"])
plt.xlabel("r [mm]")
plt.ylabel("Intensity")
plt.title(f"Radial intensity at cone end z={profiles[0]['xy_z'] * 1000.0:.3f} mm")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "radial_profile_cone_end.svg")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
for profile in profiles:
    plt.plot(profile["z"] * 1000.0, profile["axis_i"], label=profile["label"])
plt.xlabel("z [mm]")
plt.ylabel("Intensity at x=0, y=0")
plt.title("Axis intensity along z")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(OUTDIR / "axis_profile_z.svg")
plt.show()

In [ ]:
summary_lines = ["Intensity profile comparison", ""]

for profile in profiles:
    radial_peak_index = np.nanargmax(profile["radial_mean"])
    axis_peak_index = np.nanargmax(profile["axis_i"])

    summary_lines += [
        f"[{profile['label']}]",
        f"result_dir = {profile['result_dir']}",
        f"radial_xy_file = {profile['xy_file']}",
        f"radial_xy_z_m = {profile['xy_z']:.12e}",
        f"radial_peak_r_m = {profile['r'][radial_peak_index]:.12e}",
        f"radial_peak_mean_intensity = {profile['radial_mean'][radial_peak_index]:.12e}",
        f"axis_peak_z_m = {profile['z'][axis_peak_index]:.12e}",
        f"axis_peak_intensity = {profile['axis_i'][axis_peak_index]:.12e}",
        "",
    ]

(OUTDIR / "summary.txt").write_text("\n".join(summary_lines))
print("\n".join(summary_lines))